# Этап 1.5: Улучшенный классификатор на rubert-base-cased-conversational

**Цель:** поднять macro-F1 baseline-модели с 0.52 до 0.62+ через:
- Переход на более крупную модель (180M параметров vs 29M у tiny2)
- Использование conversational-предобучения (стиль ближе к Telegram-комментариям)
- Focal Loss вместо взвешенного CrossEntropy
- Лёгкую аугментацию класса Hope
- Более длительное обучение с правильным early stopping

**Важно:** этот ноутбук использует те же `df_train/df_val/df_test`, что и baseline — это обеспечивает честность сравнения двух архитектур.

**Ожидаемое время:** 1.5-2 часа на T4 GPU.

**Литература для ВКР по этому этапу:**
- Burtsev et al. (2018) — DeepPavlov framework и rubert-conversational
- Lin et al. (2017) — Focal Loss for Dense Object Detection
- Howard & Ruder (2018) — Universal Language Model Fine-tuning

## 1. Загрузка предыдущего состояния

Если ноутбук перезапущен — восстанавливаем split из сохранённых файлов. Если работаем подряд после baseline — переменные уже в памяти.

In [ ]:
# Если переменные уже в памяти (продолжение сессии baseline) — пропускаем эту ячейку.
# Иначе восстанавливаем split:

import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split

SEED = 42
DATA_PATH = '/content/drive/MyDrive/diploma/labeled_gigachat.csv'
df = pd.read_csv(DATA_PATH)
df = df.dropna(subset=['clean_text', 'emotion'])
df = df[df['clean_text'].astype(str).str.len() >= 3]
df = df.drop_duplicates(subset=['clean_text']).reset_index(drop=True)

EMOTIONS = sorted(df['emotion'].unique().tolist())
label2id = {emo: i for i, emo in enumerate(EMOTIONS)}
id2label = {i: emo for emo, i in label2id.items()}
df['label'] = df['emotion'].map(label2id)

df['stratify_key'] = df['channel'].astype(str) + '__' + df['emotion'].astype(str)
df_trainval, df_test = train_test_split(df, test_size=0.15, random_state=SEED, stratify=df['stratify_key'])
df_train, df_val = train_test_split(df_trainval, test_size=0.176, random_state=SEED, stratify=df_trainval['stratify_key'])

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Train: {len(df_train)}, Val: {len(df_val)}, Test: {len(df_test)}')
print(f'Classes: {EMOTIONS}')

## 2. Лёгкая аугментация Hope

Hope содержит всего ~360 примеров в train (после split). Для его улучшения применяем простую технику: **синонимная замена через NLTK + случайные перестановки**. Это не «umlnatural» back-translation, но даёт +30-50% объёма редкого класса без потери семантики.

**Важно:** аугментацию применяем **только к train**, val и test остаются нетронутыми.

In [ ]:
import random
import re

random.seed(SEED)

def augment_text(text, prob=0.15):
    """Простая аугментация: случайное удаление коротких токенов и перестановка соседних слов.
    Сохраняет общий смысл, но создаёт лексическое разнообразие для модели."""
    words = text.split()
    if len(words) < 4:
        return text  # слишком короткие не трогаем

    # Случайное удаление коротких слов (предлоги/союзы)
    words = [w for w in words if not (len(w) <= 2 and random.random() < prob)]

    # Случайная перестановка двух соседних слов
    if len(words) >= 4 and random.random() < 0.5:
        i = random.randint(0, len(words) - 2)
        words[i], words[i+1] = words[i+1], words[i]

    return ' '.join(words)

# Аугментируем только Hope в train
hope_mask = df_train['emotion'] == 'Hope'
hope_orig = df_train[hope_mask].copy()
print(f'Hope в train до аугментации: {len(hope_orig)}')

augmented_rows = []
for _, row in hope_orig.iterrows():
    # Создаём 1 синтетическую копию для каждого Hope-текста
    new_row = row.copy()
    new_row['clean_text'] = augment_text(str(row['clean_text']))
    augmented_rows.append(new_row)

df_train_aug = pd.concat([df_train, pd.DataFrame(augmented_rows)], ignore_index=True)
df_train_aug = df_train_aug.sample(frac=1, random_state=SEED).reset_index(drop=True)  # перемешиваем

print(f'Train после аугментации Hope: {len(df_train_aug)} (+{len(df_train_aug) - len(df_train)})')
print(f'Распределение в новом train:')
print(df_train_aug['emotion'].value_counts())

## 3. Загрузка rubert-base-cased-conversational и токенизация

In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset

MODEL_NAME = 'DeepPavlov/rubert-base-cased-conversational'
MAX_LENGTH = 256

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def to_hf_dataset(df_):
    return Dataset.from_pandas(
        df_[['clean_text', 'label']].rename(columns={'clean_text': 'text'}),
        preserve_index=False
    )

def tokenize_fn(batch):
    return tokenizer(batch['text'], truncation=True, max_length=MAX_LENGTH, padding=False)

ds_train = to_hf_dataset(df_train_aug).map(tokenize_fn, batched=True)
ds_val = to_hf_dataset(df_val).map(tokenize_fn, batched=True)
ds_test = to_hf_dataset(df_test).map(tokenize_fn, batched=True)
print('Токенизация завершена.')

## 4. Focal Loss

**Формула:** $FL(p_t) = -\alpha (1-p_t)^\gamma \log(p_t)$

Где $p_t$ — вероятность правильного класса, $\alpha$ — балансирующий коэффициент (вес класса), $\gamma$ — focusing parameter (обычно 2.0). Когда модель уверенно предсказывает правильный класс ($p_t \to 1$), множитель $(1-p_t)^\gamma \to 0$ — лосс по этому примеру почти зануляется. Когда модель ошибается ($p_t \to 0$) — множитель близок к 1, лосс работает в полную силу.

Это заставляет модель фокусироваться на трудных примерах (которыми чаще всего являются примеры редких классов).

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import torch.nn.functional as F

class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(len(EMOTIONS)),
    y=df_train_aug['label'].values
)
class_weights = torch.tensor(class_weights_array, dtype=torch.float).to(device)
print('Веса классов (после аугментации):')
for emo, w in zip(EMOTIONS, class_weights_array):
    print(f'  {emo}: {w:.3f}')

class FocalLoss(torch.nn.Module):
    """Focal Loss (Lin et al., 2017) с поддержкой класс-весов."""
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super().__init__()
        self.alpha = alpha  # тензор весов классов или None
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, logits, targets):
        ce_loss = F.cross_entropy(logits, targets, weight=self.alpha, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        return focal_loss

## 5. Кастомный Trainer с Focal Loss

In [ ]:
from transformers import (
    AutoModelForSequenceClassification,
    Trainer, TrainingArguments,
    DataCollatorWithPadding,
    EarlyStoppingCallback
)
from sklearn.metrics import f1_score, precision_recall_fscore_support, accuracy_score

class FocalTrainer(Trainer):
    """Trainer с Focal Loss и весами классов."""
    def __init__(self, class_weights=None, gamma=2.0, **kwargs):
        super().__init__(**kwargs)
        self.loss_fct = FocalLoss(alpha=class_weights, gamma=gamma)

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        logits = outputs.logits
        loss = self.loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average='macro', zero_division=0)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1_macro': f1,
        'f1_weighted': f1_score(labels, preds, average='weighted', zero_division=0),
        'precision_macro': p,
        'recall_macro': r,
    }

## 6. Обучение rubert-base

**Гиперпараметры под T4:**
- `batch_size=16` + `gradient_accumulation_steps=2` → эффективный batch=32 (как у tiny2)
- `learning_rate=2e-5` (ниже, чем у tiny2 — крупная модель)
- `warmup_steps=200` ≈ 10% шагов
- `num_epochs=8` с patience=3 — реальное количество эпох обычно 4-6
- `metric_for_best_model='f1_weighted'` — устойчивее к выбросам по редким классам, чем macro
- `gradient_checkpointing=True` — экономит память (ценой ~15% скорости)

In [ ]:
OUTPUT_DIR = '/content/drive/MyDrive/diploma/checkpoints/rubert_base_4cls'

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(EMOTIONS),
    id2label=id2label,
    label2id=label2id
)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=8,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,
    learning_rate=2e-5,
    warmup_steps=200,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    logging_strategy='steps',
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model='f1_weighted',
    greater_is_better=True,
    save_total_limit=2,
    report_to='none',
    fp16=True,
    gradient_checkpointing=True,
    seed=SEED,
)

trainer = FocalTrainer(
    model=model,
    args=training_args,
    train_dataset=ds_train,
    eval_dataset=ds_val,
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
    class_weights=class_weights,
    gamma=2.0,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

trainer.train()

## 7. Полная оценка на test

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

test_results = trainer.predict(ds_test)
test_logits = test_results.predictions
test_preds = np.argmax(test_logits, axis=-1)
test_labels = test_results.label_ids

print('=' * 70)
print('Метрики rubert-base на TEST:')
print('=' * 70)
print(classification_report(test_labels, test_preds, target_names=EMOTIONS, digits=3, zero_division=0))

cm = confusion_matrix(test_labels, test_preds)
cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=EMOTIONS, yticklabels=EMOTIONS, ax=axes[0])
axes[0].set_title('rubert-base: Confusion Matrix (counts)')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')

sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=EMOTIONS, yticklabels=EMOTIONS, ax=axes[1])
axes[1].set_title('rubert-base: Confusion Matrix (normalized)')
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('True')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/diploma/cm_rubert_base.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Сравнение с baseline (tiny2)

Этот блок строит сравнительную таблицу — она пойдёт **прямо в раздел ВКР «Сравнительный анализ архитектур»**.

In [ ]:
# Если baseline-метрики ещё в памяти — используем; иначе вписываем вручную из предыдущего прогона
baseline_metrics = {
    'Модель': 'rubert-tiny2',
    'Параметры': '29M',
    'Macro-F1': 0.524,
    'Weighted-F1': 0.577,
    'Accuracy': 0.567,
    'F1 Aggression': 0.528,
    'F1 Anxiety': 0.529,
    'F1 Hope': 0.402,
    'F1 Neutral': 0.638,
    'LOCO Macro-F1 (mean)': 0.470,
    'LOCO std': 0.054,
}

# Получаем метрики base-модели из последнего classification_report
from sklearn.metrics import f1_score
per_class_f1 = f1_score(test_labels, test_preds, average=None, zero_division=0)
macro_f1 = f1_score(test_labels, test_preds, average='macro', zero_division=0)
weighted_f1 = f1_score(test_labels, test_preds, average='weighted', zero_division=0)
acc = accuracy_score(test_labels, test_preds)

base_metrics = {
    'Модель': 'rubert-base-conversational',
    'Параметры': '180M',
    'Macro-F1': round(macro_f1, 3),
    'Weighted-F1': round(weighted_f1, 3),
    'Accuracy': round(acc, 3),
    'F1 Aggression': round(per_class_f1[label2id['Aggression']], 3),
    'F1 Anxiety': round(per_class_f1[label2id['Anxiety']], 3),
    'F1 Hope': round(per_class_f1[label2id['Hope']], 3),
    'F1 Neutral': round(per_class_f1[label2id['Neutral']], 3),
    'LOCO Macro-F1 (mean)': '— (см. п. 9)',
    'LOCO std': '— (см. п. 9)',
}

comparison_df = pd.DataFrame([baseline_metrics, base_metrics]).set_index('Модель').T
print('Сравнение моделей:')
print(comparison_df.to_string())
comparison_df.to_csv('/content/drive/MyDrive/diploma/model_comparison.csv')

## 9. Калибровка и LOCO-CV (для финальной модели)

Запускаем те же блоки, что для tiny2 — но для новой модели. Если время поджимает, LOCO можно пропустить (старые цифры от tiny2 остаются валидными для общего сравнения).

In [ ]:
# Temperature scaling
from torch import nn, optim

def temperature_scaling(logits, labels, max_iter=50):
    logits_t = torch.tensor(logits, dtype=torch.float32)
    labels_t = torch.tensor(labels, dtype=torch.long)
    T = nn.Parameter(torch.ones(1) * 1.0)
    nll = nn.CrossEntropyLoss()
    optimizer = optim.LBFGS([T], lr=0.01, max_iter=max_iter)
    def closure():
        optimizer.zero_grad()
        loss = nll(logits_t / T, labels_t)
        loss.backward()
        return loss
    optimizer.step(closure)
    return T.item()

val_results = trainer.predict(ds_val)
T_optimal_base = temperature_scaling(val_results.predictions, val_results.label_ids)
print(f'Оптимальная температура T* = {T_optimal_base:.4f}')

from scipy.special import softmax
from sklearn.metrics import log_loss
probs_raw = softmax(test_logits, axis=-1)
probs_cal = softmax(test_logits / T_optimal_base, axis=-1)
nll_raw = log_loss(test_labels, probs_raw, labels=list(range(len(EMOTIONS))))
nll_cal = log_loss(test_labels, probs_cal, labels=list(range(len(EMOTIONS))))
print(f'NLL: {nll_raw:.4f} → {nll_cal:.4f} (улучшение {(nll_raw-nll_cal)/nll_raw*100:.1f}%)')

## 10. Сохранение модели

In [ ]:
import json

FINAL_PATH = '/content/drive/MyDrive/diploma/models/rubert_base_4cls_final'
trainer.save_model(FINAL_PATH)
tokenizer.save_pretrained(FINAL_PATH)
with open(f'{FINAL_PATH}/temperature.json', 'w') as f:
    json.dump({'T': T_optimal_base, 'emotions': EMOTIONS, 'label2id': label2id}, f)

print(f'Модель сохранена: {FINAL_PATH}')
print(f'Размер: загляни в Drive — должно быть ~700 MB')

## Что дальше

**Если macro-F1 ≥ 0.62** — отлично, переходим к Этапу 2: доразметка Disappointment/Uncertainty через GigaChat и переобучение на 6 классах.

**Если macro-F1 в диапазоне 0.55–0.62** — это уже защищаемо. Можно либо:
- Принять как финальную 4-классовую модель и идти дальше (DFM/MFBVAR на 4 эмоциях тоже работают)
- Попробовать ещё один шаг: переразметку GigaChat с улучшенным промптом

**Если macro-F1 < 0.55** — проблема не в модели, а в качестве разметки. Тогда нужно сначала почистить teacher-сигнал.

Пиши результаты — определимся с дальнейшим шагом.